In [0]:
"""
Data Quality - Consistency Check Task
Checks consistency between two columns by comparing master column A with validation column B.
Supports key columns (single or composite) for grouping comparisons.
"""
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json
import re
import time

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("key_columns", "[]")
dbutils.widgets.text("column_a", "")  # Master column
dbutils.widgets.text("column_b", "")  # Validation column
dbutils.widgets.text("check_mode", "Exact Match")  # Exact Match or Case Insensitive

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
KEY_COLUMNS = json.loads(dbutils.widgets.get("key_columns"))
COLUMN_A = dbutils.widgets.get("column_a")  # Master column
COLUMN_B = dbutils.widgets.get("column_b")  # Validation column
CHECK_MODE = dbutils.widgets.get("check_mode")

# Sanitize names for table naming
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Consistency Check ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Master Column (A): {COLUMN_A}")
print(f"Validation Column (B): {COLUMN_B}")
print(f"Key Columns: {KEY_COLUMNS if KEY_COLUMNS else 'None (row-level comparison)'}")
print(f"Check Mode: {CHECK_MODE}")

# Validate inputs
if not COLUMN_A or not COLUMN_B:
    raise ValueError("Both column_a (master) and column_b (validation) must be specified")

# Read input data
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"Reading from: {input_table}")
df_input = spark.table(input_table)

total_rows = df_input.count()
print(f"Total rows: {total_rows}")

# Start with row_id only
df_checks = df_input.select("row_id")

# Create consistency check name
check_col_name = f"{COLUMN_A}_vs_{COLUMN_B}_consistency_check"
print(f"\nCreating consistency check: {check_col_name}")

# Prepare comparison expressions based on check mode
if CHECK_MODE == "Case Insensitive":
    # Case insensitive comparison (for string columns)
    col_a_expr = F.lower(F.trim(F.col(COLUMN_A).cast("string")))
    col_b_expr = F.lower(F.trim(F.col(COLUMN_B).cast("string")))
    print(f"  Using case-insensitive comparison")
else:
    # Exact match comparison
    col_a_expr = F.col(COLUMN_A)
    col_b_expr = F.col(COLUMN_B)
    print(f"  Using exact match comparison")

# Build the consistency condition
consistency_condition = col_a_expr == col_b_expr

# If key columns are specified, we can add additional grouping logic
# For now, we'll do a simple row-level comparison
# Future enhancement: aggregate by key and compare aggregated values

if KEY_COLUMNS:
    print(f"  Key columns for grouping: {KEY_COLUMNS}")
    # For now, key columns are informational - row-level comparison still applies
    # Future: Could be used to identify which groups have inconsistencies

# Create the consistency check column:
# - 1 if values match (consistent)
# - 0 if values don't match (inconsistent)
# - NULL if either column is NULL
df_with_check = df_input.select(
    "row_id",
    COLUMN_A,
    COLUMN_B
).withColumn(
    check_col_name,
    F.when(
        F.col(COLUMN_A).isNull() | F.col(COLUMN_B).isNull(),
        None  # NULL if either value is NULL
    ).when(
        consistency_condition,
        1  # Consistent (match)
    ).otherwise(
        0  # Inconsistent (no match)
    )
)

# Join back to checks dataframe
df_checks = df_checks.join(
    df_with_check.select("row_id", check_col_name),
    "row_id",
    "left"
)

# Add execution timestamp
df_checks = df_checks.withColumn("execution_timestamp", F.current_timestamp())

# Output table name
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_consistency"

print(f"\nWriting to: {output_table}")

# Write to Unity Catalog with retry logic for concurrent append exception
max_retries = 3
for attempt in range(max_retries):
    try:
        df_checks.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(output_table)
        print(f"✅ Successfully wrote {df_checks.count()} rows to {output_table}")
        break
    except Exception as e:
        if "ConcurrentAppendException" in str(e) and attempt < max_retries - 1:
            print("Concurrent append detected, retrying...")
            time.sleep(2)
        else:
            raise

# Display statistics
consistent_count = df_checks.filter(F.col(check_col_name) == 1).count()
inconsistent_count = df_checks.filter(F.col(check_col_name) == 0).count()
null_count = df_checks.filter(F.col(check_col_name).isNull()).count()
consistency_pct = (consistent_count / total_rows * 100) if total_rows > 0 else 0

print(f"\nConsistency Statistics:")
print(f"  Comparison: {COLUMN_A} == {COLUMN_B}")
print(f"  Consistent rows: {consistent_count} ({consistency_pct:.2f}%)")
print(f"  Inconsistent rows: {inconsistent_count} ({(inconsistent_count/total_rows*100) if total_rows > 0 else 0:.2f}%)")
print(f"  NULL rows: {null_count} ({(null_count/total_rows*100) if total_rows > 0 else 0:.2f}%)")
print(f"  Total rows: {total_rows}")

if KEY_COLUMNS:
    print(f"\nKey columns used for grouping: {', '.join(KEY_COLUMNS)}")